In [95]:
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.io import MemoryFile
import numpy as np
import pandas as pd
from rasterio.mask import mask
from shapely.geometry import box
import os
from mpl_toolkits.axes_grid1.anchored_artists import  AnchoredSizeBar
import matplotlib.font_manager as fm
import geopandas as gpd
base_fp = '/trace/group/rounce/cvwilson/'
site_colors = {'EC':'#63c4c7','T':'#fcc02e','Z':'#4D559C','KPS':'#BF1F6A','KQU':'#60C252'}

In [211]:
# Load DEMs and density profiles
# def load_dem(path):
#     with rasterio.open(path) as src:
#         dem = src.read(1)
#         extent = src.bounds
#     return dem, extent
vmin = 300
vmax = 4000

def load_and_plot_dem(ax, dem_path, shapefile_path, vmin=vmin, vmax=vmax, buffer_m=0, target_crs='EPSG:3338'):
    # Load and reproject shapefile
    gdf = gpd.read_file(shapefile_path).to_crs(target_crs)
    bounds = gdf.total_bounds  # [xmin, ymin, xmax, ymax]

    # Get subplot aspect ratio
    fig = ax.get_figure()
    bbox = ax.get_position()
    fig_width, fig_height = fig.get_size_inches()
    ax_width = fig_width * (bbox.x1 - bbox.x0)
    ax_height = fig_height * (bbox.y1 - bbox.y0)
    target_aspect = ax_width / ax_height

    # Adjust bounds to match subplot aspect ratio
    xmin, ymin, xmax, ymax = bounds
    xmin -= buffer_m
    xmax += buffer_m
    ymin -= buffer_m
    ymax += buffer_m
    width = xmax - xmin
    height = ymax - ymin
    current_aspect = width / height

    if current_aspect > target_aspect:
        # Too wide → increase height
        new_height = width / target_aspect
        delta = new_height - height
        ymin -= delta / 2
        ymax += delta / 2
    else:
        # Too tall → increase width
        new_width = height * target_aspect
        delta = new_width - width
        xmin -= delta / 2
        xmax += delta / 2

    crop_box = box(xmin, ymin, xmax, ymax)
    gdf_crop = gpd.GeoDataFrame(geometry=[crop_box], crs=target_crs)

    # Load and reproject DEM
    with rasterio.open(dem_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds)
        
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': target_crs,
            'transform': transform,
            'width': width,
            'height': height
        })

        dem_reprojected = np.empty((height, width), dtype=src.meta['dtype'])

        reproject(
            source=src.read(1),
            destination=dem_reprojected,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=target_crs,
            resampling=Resampling.bilinear
        )

        # Crop to adjusted bounds
        with MemoryFile() as memfile:
            with memfile.open(**kwargs) as dataset:
                dataset.write(dem_reprojected, 1)
                out_image, out_transform = mask(dataset, gdf_crop.geometry, crop=True)
                dem_cropped = out_image[0]

    # Compute extent
    h, w = dem_cropped.shape
    x_min = out_transform.c
    y_max = out_transform.f
    x_max = x_min + out_transform.a * w
    y_min = y_max + out_transform.e * h
    extent = [x_min, x_max, y_min, y_max]

    # Plot
    ax.imshow(dem_cropped, extent=extent, cmap='terrain', origin='upper', vmin=vmin, vmax=vmax)
    # ax.axis('off')
    gdf.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1)
    left = 0.1 if 'kahiltna' in shapefile_path else 0.1
    up = 0.085 if 'kahiltna' in shapefile_path else 0.2
    up_text = 0.04 if 'kahiltna' in shapefile_path else 0.1
    ax.annotate('N', xy=(left, up), xytext=(left, up_text),
            xycoords='axes fraction', textcoords='axes fraction',
            ha='center', va='center', fontsize=12, fontweight='bold',
            arrowprops=dict(arrowstyle='-|>', facecolor='black', lw=0, mutation_scale=20))
    # for spine in ax.spines.values():
    #     spine.set_edgecolor('black')
    #     spine.set_linewidth(2)
    ax.ticklabel_format(style='scientific', axis='x', scilimits=(0,0))
    length_m = 100000 if 'kahiltna' in shapefile_path else 10000
    add_scale_bar(ax, out_transform, length_m)
    
    return

# Plot density profile
def plot_density(ax, depth, density, title, site):
    ax.plot(density, depth, color=site_colors[site])
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel('Density (kg/m³)')
    ax.set_ylabel('Depth (m)')

def adjust_bounds_to_aspect(bounds, target_aspect, buffer_m=0):
    xmin, ymin, xmax, ymax = bounds
    width = xmax - xmin
    height = ymax - ymin

    # Apply buffer
    xmin -= buffer_m
    xmax += buffer_m
    ymin -= buffer_m
    ymax += buffer_m

    # Recalculate with buffer
    width = xmax - xmin
    height = ymax - ymin
    current_aspect = width / height

    # Adjust bounds to match target aspect ratio
    if current_aspect > target_aspect:
        # Too wide → increase height
        new_height = width / target_aspect
        delta = new_height - height
        ymin -= delta / 2
        ymax += delta / 2
    else:
        # Too tall → increase width
        new_width = height * target_aspect
        delta = new_width - width
        xmin -= delta / 2
        xmax += delta / 2

    return box(xmin, ymin, xmax, ymax)

import matplotlib.font_manager as fm

def add_scale_bar(ax, transform, length_m=10000, location='lower right', linewidth=2, text=None):
    # Get pixel size from affine transform
    pixel_size = transform.a  # meters per pixel (assuming square pixels)

    # Compute scale bar length in pixels
    length_px = length_m / pixel_size

    # Optional label
    label = text if text else f'{length_m/1000:.0f} km'

    # Add scale bar
    fontprops = fm.FontProperties(size=14)
    scalebar = AnchoredSizeBar(ax.transData,
                                length_px, label, location,
                                pad=0.5, color='black', frameon=False,
                                size_vertical=linewidth, fontproperties=fontprops)
    ax.add_artist(scalebar)

def add_elevation_colorbar(ax_legend, cmap='terrain', vmin=300, vmax=4000, label='Elevation (m a.s.l.)', width_frac=0.1):
    box = ax_legend.get_position()
    ax_legend.set_position([box.x0, box.y0, box.width * width_frac, box.height*0.9])

    from matplotlib.cm import ScalarMappable
    sm = ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])  # dummy array

    cb = plt.colorbar(sm, cax=ax_legend, orientation='vertical')
    cb.set_label(label)

import pyproj

def plot_latlon(ax, lat, lon, site, crs='EPSG:3338', marker_size=50):
    # Transform lat/lon to projected coordinates
    transformer = pyproj.Transformer.from_crs("EPSG:4326", crs, always_xy=True)
    x, y = transformer.transform(lon, lat)

    # Plot the point
    ax.scatter(x, y, s=marker_size, color='k', zorder=10)
    ax.annotate(site, xy=(x, y), xytext=(13, -13),
            textcoords='offset points', fontsize=16,fontweight='bold',
            # bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', boxstyle='round,pad=0.2'),
            ha='center', va='center', color='k',
            arrowprops=None)


In [212]:
# Load data
density_fp = '/trace/home/cvwilson/INTERN/CommunityFirnModel/Data/cores/'
wfiles = os.listdir(density_fp + 'wolverine')
wdfs = []
for wf in wfiles:
    if 'csv' in wf and 'snow' not in wf and (('_04_' in wf) or ('_05_' in wf)):
        df = pd.read_csv(density_fp + 'wolverine/'+wf)
        wdfs.append(df)
gzdfs = [pd.read_csv(density_fp + 'gulkana/gulkanaZ_2025_04_20.csv')]
gtdfs = [pd.read_csv(density_fp + 'gulkana/gulkanaT_2025_04_20.csv')]
kpsdfs = [pd.read_csv(density_fp + 'kahiltna/kahiltnaKPS_2024_05_26.csv'),
         pd.read_csv(density_fp + 'kahiltna/kahiltnaKPS_2025_05_23.csv')]
kqudfs = [pd.read_csv(density_fp + 'kahiltna/kahiltnaKQU_2024_05_25.csv'),
          pd.read_csv(density_fp + 'kahiltna/kahiltnaKQU_2025_05_23.csv')]
all_df = {'EC':wdfs, 'Z':gzdfs, 'T':gtdfs, 'KPS':kpsdfs, 'KQU':kqudfs}

In [214]:
# # Create figure and layout
# fig = plt.figure(figsize=(16, 8))
# gs = gridspec.GridSpec(2, 4, width_ratios=[1, 1.6, 0.98, 1], height_ratios=[1, 1], hspace=0.2, wspace=0.3)

# # Column 0: Density profiles
# ax_density_w = fig.add_subplot(gs[0, 0])
# ax_density_g = fig.add_subplot(gs[1, 0])

# # Column 1: DEMs
# ax_dem_w = fig.add_subplot(gs[0, 1])
# ax_dem_g = fig.add_subplot(gs[1, 1])

# # Column 2: Kahiltna DEM (both rows)
# ax_dem_k = fig.add_subplot(gs[:, 2])

# # Column 3: Kahiltna density and legend
# ax_density_k = fig.add_subplot(gs[0, 3])
# ax_legend = fig.add_subplot(gs[1, 3])

# # Plot density profiles
# dens_axes = {'gulkana':ax_density_g, 'wolverine':ax_density_w, 'kahiltna':ax_density_k}
# dem_axes = {'gulkana':ax_dem_g, 'wolverine':ax_dem_w, 'kahiltna':ax_dem_k}

# ymax = {'gulkana':16.2, 'kahiltna':16.5, 'wolverine':28}
# site_loc = {'EC':(60.42594,-148.914234), 'KPS':(63.076633856,-151.174260401),
#             'KQU':(62.976647,-151.070586),
#              'Z':(63.288623,-145.482731), 'T':(63.279063,-145.461041)}
# for site in all_df:
#     glacier = 'wolverine' if site == 'EC' else 'kahiltna' if 'K' in site else 'gulkana'
#     ax = dens_axes[glacier]
#     for df in all_df[site]:
#         plot_density(ax, df['SBD'], df['density'], '', site=site)
#     ax.plot(np.nan, np.nan, color=site_colors[site], label=site)
#     ax.legend()
# for glacier in dens_axes:
#     dens_axes[glacier].set_ylim(0, ymax[glacier])
#     dens_axes[glacier].invert_yaxis()
#     dens_axes[glacier].set_xlim(200, 900)

# # Plot DEMs
# dem_fp = base_fp + 'dems/'
# load_and_plot_dem(ax_dem_w, dem_fp + 'wolverine_dem.tif', dem_fp + 'wolverine_shapefile.shp')
# load_and_plot_dem(ax_dem_g, dem_fp + 'gulkana_dem.tif', dem_fp + 'gulkana_shapefile.shp')
# load_and_plot_dem(ax_dem_k, dem_fp + 'kahiltna_dem.tif', dem_fp + 'kahiltna_shapefile.shp')
# for site in all_df:
#     glacier = 'wolverine' if site == 'EC' else 'kahiltna' if 'K' in site else 'gulkana'
#     plot_latlon(dem_axes[glacier], site_loc[site][0], site_loc[site][1], site)

# # Legend placeholder
# add_elevation_colorbar(ax_legend)

# for ax in fig.axes:
#     ax.xaxis.label.set_fontsize(12)
#     ax.yaxis.label.set_fontsize(12)
#     ax.tick_params(axis='both', labelsize=12)
#     ax.xaxis.offsetText.set_fontsize(12)
#     ax.yaxis.offsetText.set_fontsize(12)


# # plt.tight_layout()
# plt.savefig(base_fp + 'Firn/Figs/big_fig.png', dpi=300, bbox_inches='tight')
# plt.show()